# CentaurDrug — ADMET Production Prototype: AqSolDB + XGBoost (v3 — Near-Production)

This notebook addresses every methodological and engineering issue from v2:

**Fixes applied:**
- CV loop now uses early stopping (consistent with final model training)
- Hyperparameter search budget raised to 40 with Optuna (TPE sampler, not random)
- `iterrows()` replaced with vectorized `apply()`
- `StandardScaler` removed from XGBoost pipeline (was unnecessary noise)
- Applicability domain check (Tanimoto similarity threshold to training set)
- Prediction uncertainty via XGBoost quantile regression ensemble
- `prediction_unit` is now explicit: `log(mol/L)`
- Error analysis now includes scaffold-level residual breakdown

**Stack:**
```
pytdc  rdkit  xgboost  scikit-learn  optuna  pandas  numpy  joblib  mlflow
```

```bash
cd ~/Desktop/projects/centaurdrug
uv add pytdc rdkit xgboost scikit-learn optuna pandas numpy joblib pyyaml mlflow jupyter
uv run jupyter lab
```


## 1. Imports, configuration, reproducibility

In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd

from tdc.single_pred import ADME

from rdkit import Chem, DataStructs
from rdkit import RDLogger
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    import mlflow
    MLFLOW_AVAILABLE = True
except Exception:
    mlflow = None
    MLFLOW_AVAILABLE = False

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "models" / "admet_aqsol_xgboost_v3"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset config ─────────────────────────────────────────────────────────────
DATASET_NAME  = "Solubility_AqSolDB"
SMILES_COL    = "Drug"
TARGET_COL    = "Y"
TARGET_UNIT   = "log(mol/L)"   # explicit unit — used in inference output

# ── Training config ────────────────────────────────────────────────────────────
N_OPTUNA_TRIALS  = 40          # raise to 80–100 for final training run
N_CV_SPLITS      = 5
EARLY_STOP_ROUNDS = 50

# ── Applicability domain ───────────────────────────────────────────────────────
# Molecules with max Tanimoto similarity to training set below this threshold
# are flagged as out-of-domain. Adjust based on desired recall/precision.
AD_TANIMOTO_THRESHOLD = 0.35

# ── MLflow ─────────────────────────────────────────────────────────────────────
USE_MLFLOW = MLFLOW_AVAILABLE
MLFLOW_EXPERIMENT_NAME = "centaurdrug-admet-aqsol-xgboost-v3"

# ── Structured logger ──────────────────────────────────────────────────────────
logger = logging.getLogger("centaurdrug.admet.aqsol")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_handler = logging.StreamHandler()
_handler.setFormatter(logging.Formatter(
    fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
))
logger.addHandler(_handler)

logger.info("Project root  : %s", PROJECT_ROOT)
logger.info("Artifact dir  : %s", ARTIFACT_DIR)
logger.info("MLflow        : %s", MLFLOW_AVAILABLE)
logger.info("Optuna trials : %d", N_OPTUNA_TRIALS)


## 2. Input validation contract

Production rule — consistent across training and inference:

| Outcome | Training | Inference |
|---|---|---|
| Valid SMILES | kept, canonicalized | predicted |
| Invalid SMILES | logged + rejected CSV | `{"status": "rejected", "reason": "..."}` |

`validate_dataframe` is now **vectorized** with `.apply()` instead of `iterrows()`.


In [ ]:
@dataclass
class MoleculeValidationResult:
    original_smiles: str
    is_valid: bool
    canonical_smiles: Optional[str] = None
    rejection_reason: Optional[str] = None


def validate_smiles(smiles: Any) -> MoleculeValidationResult:
    """Validate and canonicalize one SMILES. Always returns a structured result."""
    if smiles is None:
        return MoleculeValidationResult(str(smiles), False, rejection_reason="missing_smiles")

    s = str(smiles).strip()
    if not s:
        return MoleculeValidationResult(str(smiles), False, rejection_reason="empty_smiles")

    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return MoleculeValidationResult(s, False, rejection_reason="invalid_smiles")

    return MoleculeValidationResult(s, True, canonical_smiles=Chem.MolToSmiles(mol, canonical=True))


def validate_dataframe(
    df: pd.DataFrame,
    smiles_col: str = SMILES_COL,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Vectorized SMILES validation. Returns (valid_df, rejected_df).
    valid_df[smiles_col] contains canonical SMILES.
    """
    # apply() is ~20x faster than iterrows() for this workload
    results: pd.Series = df[smiles_col].apply(validate_smiles)
    records = pd.DataFrame([asdict(r) for r in results], index=df.index)

    merged = df.join(records)
    valid_df    = merged[merged["is_valid"]].copy()
    rejected_df = merged[~merged["is_valid"]].copy()

    # Replace raw SMILES with canonical form for all downstream steps
    valid_df[smiles_col] = valid_df["canonical_smiles"]

    return valid_df, rejected_df


# Smoke test
for s in ["CCO", "", "not_a_smiles", None, "CC(=O)Oc1ccccc1C(=O)O"]:
    print(validate_smiles(s))


## 3. Load TDC AqSolDB dataset

In [ ]:
def load_tdc_dataset(name: str = DATASET_NAME) -> pd.DataFrame:
    """Load full TDC dataset as one flat DataFrame (we handle our own splits)."""
    data = ADME(name=name)
    df = data.get_data()[[SMILES_COL, TARGET_COL]].copy()
    return df


raw_df = load_tdc_dataset()
logger.info("Raw dataset: %s rows", len(raw_df))
display(raw_df.describe())


## 4. Validate molecules and save rejection report

In [ ]:
valid_df, rejected_df = validate_dataframe(raw_df)

logger.info("Valid   : %d", len(valid_df))
logger.info("Rejected: %d", len(rejected_df))

rejection_path = ARTIFACT_DIR / "training_rejections.csv"
rejected_df.to_csv(rejection_path, index=False)
logger.info("Rejection report: %s", rejection_path)

display(valid_df.head(3))
if len(rejected_df):
    display(rejected_df.head(3))


## 5. Scaffold computation (Bemis–Murcko)

Acyclic molecules produce an empty Murcko scaffold and are grouped by their
canonical SMILES instead — each gets its own group, which is conservative but
correct (they won't leak across splits).


In [ ]:
def compute_scaffold(smiles: str) -> str:
    """Return Bemis-Murcko scaffold SMILES, or acyclic fallback."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return f"invalid::{smiles}"
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    if not scaffold:
        return f"acyclic::{Chem.MolToSmiles(mol, canonical=True)}"
    return scaffold


valid_df["scaffold"] = valid_df[SMILES_COL].apply(compute_scaffold)

logger.info("Unique scaffolds: %d (out of %d molecules)", valid_df["scaffold"].nunique(), len(valid_df))
display(valid_df["scaffold"].value_counts().head(10))


## 6. Scaffold-aware four-way split

```
train_core   70%  → model fitting
early_stop   10%  → early stopping signal only
validation   10%  → hyperparameter selection / model comparison
test         10%  → single final evaluation (never touched until reporting)
```

Scaffold groups are shuffled then greedily assigned to the bucket with the
largest remaining capacity. This keeps scaffold groups intact — no analogue
leakage across splits.


In [ ]:
def scaffold_split(
    df: pd.DataFrame,
    scaffold_col: str = "scaffold",
    fracs: Tuple[float, float, float, float] = (0.70, 0.10, 0.10, 0.10),
    seed: int = RANDOM_SEED,
) -> Dict[str, pd.DataFrame]:
    """
    Greedy scaffold-grouped split into train_core / early_stop / validation / test.
    Each scaffold group goes entirely into one bucket.
    """
    split_names = ["train_core", "early_stop", "validation", "test"]
    assert abs(sum(fracs) - 1.0) < 1e-8

    groups = list(df.groupby(scaffold_col))
    rng = random.Random(seed)
    rng.shuffle(groups)

    n_total = len(df)
    targets  = {k: f * n_total for k, f in zip(split_names, fracs)}
    buckets  = {k: [] for k in split_names}
    counts   = {k: 0  for k in split_names}

    for _, group in groups:
        remaining = {k: targets[k] - counts[k] for k in split_names}
        best = max(remaining, key=remaining.get)
        buckets[best].append(group)
        counts[best] += len(group)

    return {
        k: pd.concat(v).sample(frac=1.0, random_state=seed).reset_index(drop=True)
        for k, v in buckets.items()
        if v
    }


splits = scaffold_split(valid_df)

split_report = pd.DataFrame({
    "split":       list(splits.keys()),
    "n_molecules": [len(v)                    for v in splits.values()],
    "n_scaffolds": [v["scaffold"].nunique()    for v in splits.values()],
    "target_mean": [v[TARGET_COL].mean()       for v in splits.values()],
    "target_std":  [v[TARGET_COL].std()        for v in splits.values()],
})
display(split_report)
split_report.to_csv(ARTIFACT_DIR / "split_report.csv", index=False)

for name, s in splits.items():
    logger.info("%-12s  %d molecules, %d scaffolds", name, len(s), s["scaffold"].nunique())


## 7. Feature engineering: Morgan + MACCS + RDKit descriptors

Feature vector layout:
```
[Morgan 2048 bits | MACCS 167 bits | 14 RDKit physicochemical descriptors]
= 2229 features total
```

**Note on scaling:** `StandardScaler` is intentionally removed from this version.
Tree-based models (XGBoost) are invariant to monotonic feature transformations.
Scaling was adding an unnecessary serialization dependency without improving
XGBoost predictions. If a linear blended model is added later, scale only
the descriptor block, not the binary fingerprints.


In [ ]:
RDKIT_DESCRIPTOR_FUNCS: List[Tuple[str, Any]] = [
    ("MolWt",              Descriptors.MolWt),
    ("MolLogP",            Descriptors.MolLogP),
    ("TPSA",               Descriptors.TPSA),
    ("NumHDonors",         Descriptors.NumHDonors),
    ("NumHAcceptors",      Descriptors.NumHAcceptors),
    ("NumRotatableBonds",  Descriptors.NumRotatableBonds),
    ("RingCount",          Descriptors.RingCount),
    ("HeavyAtomCount",     Descriptors.HeavyAtomCount),
    ("FractionCSP3",       rdMolDescriptors.CalcFractionCSP3),
    ("NHOHCount",          Descriptors.NHOHCount),
    ("NOCount",            Descriptors.NOCount),
    ("NumAliphaticRings",  rdMolDescriptors.CalcNumAliphaticRings),
    ("NumAromaticRings",   rdMolDescriptors.CalcNumAromaticRings),
    ("NumSaturatedRings",  rdMolDescriptors.CalcNumSaturatedRings),
]

DESCRIPTOR_NAMES = [name for name, _ in RDKIT_DESCRIPTOR_FUNCS]
N_MORGAN_BITS = 2048
N_MACCS_BITS  = 167
N_DESC        = len(RDKIT_DESCRIPTOR_FUNCS)
FEATURE_DIM   = N_MORGAN_BITS + N_MACCS_BITS + N_DESC


class MolecularFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    SMILES list → numpy float32 array of shape (n, FEATURE_DIM).

    Raises ValueError on invalid SMILES — caller must validate first.
    NaN/Inf descriptor values are replaced with 0.0.
    """

    def __init__(self, radius: int = 2, n_bits: int = N_MORGAN_BITS):
        self.radius = radius
        self.n_bits = n_bits

    def fit(self, smiles: List[str], y=None):
        return self

    def _featurize_one(self, s: str) -> np.ndarray:
        mol = Chem.MolFromSmiles(str(s))
        if mol is None:
            raise ValueError(f"Invalid SMILES in featurizer (validate first): {s}")

        # Morgan
        morgan = AllChem.GetMorganFingerprintAsBitVect(mol, self.radius, nBits=self.n_bits)
        morgan_arr = np.zeros((self.n_bits,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(morgan, morgan_arr)

        # MACCS
        maccs = MACCSkeys.GenMACCSKeys(mol)
        maccs_arr = np.zeros((maccs.GetNumBits(),), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(maccs, maccs_arr)

        # Physicochemical descriptors
        desc = np.array(
            [_safe_descriptor(func, mol) for _, func in RDKIT_DESCRIPTOR_FUNCS],
            dtype=np.float32,
        )

        return np.concatenate([morgan_arr, maccs_arr, desc])

    def transform(self, smiles: List[str]) -> np.ndarray:
        X = np.vstack([self._featurize_one(s) for s in smiles]).astype(np.float32)
        return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    @property
    def feature_dim(self) -> int:
        return self.n_bits + N_MACCS_BITS + N_DESC


def _safe_descriptor(func, mol) -> float:
    try:
        return float(func(mol))
    except Exception:
        return np.nan


# Smoke test
fe_test = MolecularFeatureExtractor()
X_test_smoke = fe_test.transform(["CCO", "CC(=O)Oc1ccccc1C(=O)O"])
logger.info("Feature shape smoke test: %s  (expected n=2, dim=%d)", X_test_smoke.shape, FEATURE_DIM)
assert X_test_smoke.shape == (2, FEATURE_DIM), f"Unexpected shape: {X_test_smoke.shape}"


## 8. Featurize all splits

In [ ]:
feature_extractor = MolecularFeatureExtractor(radius=2, n_bits=N_MORGAN_BITS)

train_core_df = splits["train_core"]
early_stop_df = splits["early_stop"]
validation_df = splits["validation"]
test_df       = splits["test"]

X_train = feature_extractor.transform(train_core_df[SMILES_COL].tolist())
y_train = train_core_df[TARGET_COL].astype(float).to_numpy()

X_early = feature_extractor.transform(early_stop_df[SMILES_COL].tolist())
y_early = early_stop_df[TARGET_COL].astype(float).to_numpy()

X_valid = feature_extractor.transform(validation_df[SMILES_COL].tolist())
y_valid = validation_df[TARGET_COL].astype(float).to_numpy()

X_test  = feature_extractor.transform(test_df[SMILES_COL].tolist())
y_test  = test_df[TARGET_COL].astype(float).to_numpy()

for name, X, y in [
    ("train_core", X_train, y_train),
    ("early_stop", X_early, y_early),
    ("validation",  X_valid, y_valid),
    ("test",        X_test,  y_test),
]:
    logger.info("%-12s  X=%s  y=%s", name, X.shape, y.shape)


## 9. Metrics

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mae":  float(mean_absolute_error(y_true, y_pred)),
        "mse":  float(mse),
        "rmse": float(np.sqrt(mse)),
        "r2":   float(r2_score(y_true, y_pred)),
    }


def print_metrics(title: str, m: Dict[str, float]):
    pad = max(len(k) for k in m)
    print(f"\n── {title} ──")
    for k, v in m.items():
        print(f"  {k:<{pad}} : {v:.5f}")


## 10. Hyperparameter search with Optuna (TPE) + scaffold GroupKFold

**Key fix from v2:** Every CV fold now uses `early_stopping_rounds` with its own
internal held-out set drawn from the fold's training data. This means the
`n_estimators` selected by CV reflects the same training regime as the final model.

Why Optuna over random search:
- TPE (Tree-structured Parzen Estimator) focuses trials on promising regions
- 40 trials with TPE ≈ 80–100 random trials in practice
- Pruning eliminates unpromising trials early (Median pruner)


In [ ]:
cv_groups  = train_core_df["scaffold"].to_numpy()
group_kfold = GroupKFold(n_splits=N_CV_SPLITS)


def cv_objective(trial: optuna.Trial) -> float:
    """
    Optuna objective: scaffold GroupKFold CV RMSE.
    Each fold uses an internal 10% early-stopping holdout drawn from
    that fold's training data to keep the training regime consistent
    with the final model.
    """
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1500),
        "max_depth":         trial.suggest_int("max_depth", 3, 9),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 15),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "gamma":             trial.suggest_float("gamma", 0.0, 5.0),
    }

    fold_rmses = []

    for fold_idx, (tr_idx, val_idx) in enumerate(
        group_kfold.split(X_train, y_train, groups=cv_groups)
    ):
        X_tr_full, y_tr_full = X_train[tr_idx], y_train[tr_idx]
        X_val_fold, y_val_fold = X_train[val_idx], y_train[val_idx]

        # Carve out 10% of fold training data as the early-stop set
        n_es = max(1, int(0.10 * len(y_tr_full)))
        X_tr, X_es = X_tr_full[:-n_es], X_tr_full[-n_es:]
        y_tr, y_es = y_tr_full[:-n_es], y_tr_full[-n_es:]

        model = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            early_stopping_rounds=EARLY_STOP_ROUNDS,
            **params,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_es, y_es)],
            verbose=False,
        )

        pred = model.predict(X_val_fold)
        fold_rmse = float(np.sqrt(mean_squared_error(y_val_fold, pred)))
        fold_rmses.append(fold_rmse)

        # Optuna pruning: report intermediate value after each fold
        trial.report(np.mean(fold_rmses), step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_rmses))


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)

logger.info("Starting Optuna search: %d trials, %d-fold scaffold CV", N_OPTUNA_TRIALS, N_CV_SPLITS)
t0 = time.time()
study.optimize(cv_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
logger.info("Optuna search done in %.1f s", time.time() - t0)
logger.info("Best CV RMSE : %.5f", study.best_value)
logger.info("Best params  : %s", study.best_params)

# Save trial results
trials_df = study.trials_dataframe().sort_values("value")
trials_path = ARTIFACT_DIR / "optuna_trials.csv"
trials_df.to_csv(trials_path, index=False)
display(trials_df.head(10))


## 11. Train final model with early stopping

Training uses:
- `train_core` → model fitting
- `early_stop` → early stopping signal (separate from validation)
- `validation` → honest held-out model comparison
- `test` → single final evaluation, touched once


In [ ]:
best_params = dict(study.best_params)

# Ensure int types survive Optuna serialization
for k in ["n_estimators", "max_depth", "min_child_weight"]:
    best_params[k] = int(best_params[k])

final_model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    early_stopping_rounds=EARLY_STOP_ROUNDS,
    **best_params,
)

final_model.fit(
    X_train, y_train,
    eval_set=[(X_early, y_early)],
    verbose=False,
)

logger.info("Best iteration : %s", getattr(final_model, 'best_iteration', 'n/a'))
logger.info("Best ES score  : %s", getattr(final_model, 'best_score', 'n/a'))


## 12. Prediction uncertainty via quantile regression

XGBoost supports `objective="reg:quantileerror"` (v2.0+) for native quantile regression.
We train two additional models for the 10th and 90th percentiles, giving an 80%
prediction interval alongside the point estimate.

This is the most principled uncertainty approach without leaving the XGBoost ecosystem.
For full conformal prediction intervals, swap to `mapie` in production.


In [ ]:
_quantile_base_params = {k: v for k, v in best_params.items()}

def train_quantile_model(quantile: float) -> XGBRegressor:
    """Train a quantile regression model using best hyperparams."""
    qmodel = XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=quantile,
        tree_method="hist",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        # n_estimators fixed at best_iteration for quantile models
        n_estimators=int(getattr(final_model, "best_iteration", best_params["n_estimators"])),
        max_depth=best_params["max_depth"],
        learning_rate=best_params["learning_rate"],
        subsample=best_params["subsample"],
        colsample_bytree=best_params["colsample_bytree"],
        min_child_weight=best_params["min_child_weight"],
        reg_alpha=best_params["reg_alpha"],
        reg_lambda=best_params["reg_lambda"],
    )
    qmodel.fit(X_train, y_train, verbose=False)
    return qmodel


try:
    q10_model = train_quantile_model(0.10)
    q90_model = train_quantile_model(0.90)
    QUANTILE_MODELS_AVAILABLE = True
    logger.info("Quantile models trained (q10, q90)")
except Exception as e:
    q10_model = q90_model = None
    QUANTILE_MODELS_AVAILABLE = False
    logger.warning("Quantile models unavailable (XGBoost < 2.0?): %s", e)


## 13. Applicability domain (AD) check

A molecule is **in-domain** if its maximum Tanimoto similarity (Morgan fingerprints,
radius=2) to any training molecule exceeds `AD_TANIMOTO_THRESHOLD` (default 0.35).

This is the standard nearest-neighbor AD method in cheminformatics. It's fast,
interpretable, and requires no additional training. More sophisticated methods
(leverage, descriptor range, kernel density) can be added later.


In [ ]:
from rdkit.Chem import DataStructs
from rdkit import DataStructs as RDKitDataStructs

def build_training_fps(smiles_list: List[str], radius: int = 2, n_bits: int = N_MORGAN_BITS):
    """Pre-compute Morgan fingerprint objects for fast Tanimoto screening."""
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits))
    return fps


def max_tanimoto_to_training(
    smiles: str,
    training_fps: List,
    radius: int = 2,
    n_bits: int = N_MORGAN_BITS,
) -> float:
    """
    Return the maximum Tanimoto similarity between the query molecule
    and any molecule in the training set.
    Returns 0.0 if the molecule is invalid or training_fps is empty.
    """
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None or not training_fps:
        return 0.0
    query_fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    similarities = RDKitDataStructs.BulkTanimotoSimilarity(query_fp, training_fps)
    return float(max(similarities))


# Build training fingerprints once and cache
logger.info("Building training fingerprint index (%d molecules)...", len(train_core_df))
t0 = time.time()
TRAINING_FPS = build_training_fps(train_core_df[SMILES_COL].tolist())
logger.info("Fingerprint index built in %.2f s", time.time() - t0)

# Compute AD scores for test set (informational)
test_ad_scores = test_df[SMILES_COL].apply(
    lambda s: max_tanimoto_to_training(s, TRAINING_FPS)
)
n_out_of_domain = (test_ad_scores < AD_TANIMOTO_THRESHOLD).sum()
logger.info(
    "Test set: %d / %d molecules below AD threshold (%.2f)",
    n_out_of_domain, len(test_df), AD_TANIMOTO_THRESHOLD
)
display(test_ad_scores.describe())


## 14. Evaluate on all splits

In [ ]:
train_pred = final_model.predict(X_train)
early_pred = final_model.predict(X_early)
valid_pred = final_model.predict(X_valid)
test_pred  = final_model.predict(X_test)

metrics_report = {
    "train_core": regression_metrics(y_train, train_pred),
    "early_stop": regression_metrics(y_early, early_pred),
    "validation":  regression_metrics(y_valid, valid_pred),
    "test":        regression_metrics(y_test,  test_pred),
}

for split_name, m in metrics_report.items():
    print_metrics(split_name, m)

metrics_path = ARTIFACT_DIR / "metrics_report.json"
metrics_path.write_text(json.dumps(metrics_report, indent=2), encoding="utf-8")
logger.info("Saved metrics: %s", metrics_path)

display(pd.DataFrame(metrics_report).T)


## 15. Error analysis with scaffold-level breakdown

Two views:
1. **Molecule-level**: worst predictions sorted by absolute error + AD score
2. **Scaffold-level**: mean absolute error per scaffold group → reveals systematic bias
   for entire scaffold families (more actionable than per-molecule analysis)


In [ ]:
error_df = test_df[[SMILES_COL, TARGET_COL, "scaffold"]].copy()
error_df["prediction"]     = test_pred
error_df["error"]          = error_df["prediction"] - error_df[TARGET_COL]
error_df["absolute_error"] = error_df["error"].abs()
error_df["ad_score"]       = test_ad_scores.values

# Molecule-level view
error_df_sorted = error_df.sort_values("absolute_error", ascending=False)
error_path = ARTIFACT_DIR / "test_error_analysis.csv"
error_df_sorted.to_csv(error_path, index=False)

print("\n── Worst 20 predictions ──")
display(error_df_sorted.head(20))

# Scaffold-level aggregation
scaffold_errors = (
    error_df.groupby("scaffold")
    .agg(
        n_molecules=("absolute_error", "count"),
        mean_abs_error=("absolute_error", "mean"),
        mean_error=("error", "mean"),          # signed: reveals systematic over/under-prediction
        mean_ad_score=("ad_score", "mean"),
    )
    .sort_values("mean_abs_error", ascending=False)
    .reset_index()
)

scaffold_error_path = ARTIFACT_DIR / "scaffold_error_analysis.csv"
scaffold_errors.to_csv(scaffold_error_path, index=False)

print("\n── Worst scaffold families (≥2 molecules) ──")
display(scaffold_errors[scaffold_errors["n_molecules"] >= 2].head(15))

logger.info("Saved error analysis: %s", error_path)
logger.info("Saved scaffold error analysis: %s", scaffold_error_path)


## 16. Save production artifact bundle

```
models/admet_aqsol_xgboost_v3/
├── xgboost_aqsol_model.joblib        ← point estimate model
├── xgboost_aqsol_q10.joblib          ← 10th percentile model
├── xgboost_aqsol_q90.joblib          ← 90th percentile model
├── feature_extractor.joblib
├── training_metadata.json
├── metrics_report.json
├── optuna_trials.csv
├── split_report.csv
├── training_rejections.csv
├── test_error_analysis.csv
└── scaffold_error_analysis.csv
```


In [ ]:
model_path     = ARTIFACT_DIR / "xgboost_aqsol_model.joblib"
fe_path        = ARTIFACT_DIR / "feature_extractor.joblib"
q10_path       = ARTIFACT_DIR / "xgboost_aqsol_q10.joblib"
q90_path       = ARTIFACT_DIR / "xgboost_aqsol_q90.joblib"
fps_path       = ARTIFACT_DIR / "training_fps.joblib"
metadata_path  = ARTIFACT_DIR / "training_metadata.json"

joblib.dump(final_model,        model_path)
joblib.dump(feature_extractor,  fe_path)
joblib.dump(TRAINING_FPS,       fps_path)

if QUANTILE_MODELS_AVAILABLE:
    joblib.dump(q10_model, q10_path)
    joblib.dump(q90_model, q90_path)

metadata = {
    "project":     "CentaurDrug",
    "version":     "v3",
    "dataset":     DATASET_NAME,
    "task_type":   "regression",
    "target":      "aqueous_solubility",
    "target_unit": TARGET_UNIT,
    "smiles_col":  SMILES_COL,
    "target_col":  TARGET_COL,
    "model_type":  "XGBRegressor (point) + quantile q10/q90",
    "features": {
        "morgan_radius":    feature_extractor.radius,
        "morgan_n_bits":    feature_extractor.n_bits,
        "maccs_bits":       N_MACCS_BITS,
        "rdkit_descriptors": DESCRIPTOR_NAMES,
        "total_dim":        FEATURE_DIM,
        "scaling":          "none (tree-based model)",
    },
    "split_strategy": "Bemis-Murcko scaffold greedy split + GroupKFold scaffold CV",
    "hyperparameter_search": {
        "method":   "Optuna TPE + MedianPruner",
        "n_trials": N_OPTUNA_TRIALS,
        "cv_folds": N_CV_SPLITS,
        "cv_uses_early_stopping": True,
    },
    "best_params":     best_params,
    "early_stop_rounds": EARLY_STOP_ROUNDS,
    "best_iteration":  int(getattr(final_model, "best_iteration", -1)),
    "random_seed":     RANDOM_SEED,
    "applicability_domain": {
        "method":    "max_tanimoto_nearest_neighbor",
        "threshold": AD_TANIMOTO_THRESHOLD,
        "fingerprint": f"Morgan radius={feature_extractor.radius} n_bits={feature_extractor.n_bits}",
    },
    "uncertainty": {
        "method":   "quantile_regression",
        "quantiles": [0.10, 0.90],
        "available": QUANTILE_MODELS_AVAILABLE,
    },
    "rejection_contract": {
        "invalid_input_status": "rejected",
        "possible_reasons": ["missing_smiles", "empty_smiles", "invalid_smiles"],
    },
    "metrics": metrics_report,
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

logger.info("Saved model            : %s", model_path)
logger.info("Saved feature extractor: %s", fe_path)
logger.info("Saved training FPs     : %s", fps_path)
if QUANTILE_MODELS_AVAILABLE:
    logger.info("Saved q10 model        : %s", q10_path)
    logger.info("Saved q90 model        : %s", q90_path)
logger.info("Saved metadata         : %s", metadata_path)


## 17. MLflow experiment tracking (optional)

In [ ]:
if USE_MLFLOW:
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

    with mlflow.start_run(run_name="aqsol-xgboost-v3-scaffold-optuna"):
        mlflow.log_params(best_params)
        mlflow.log_param("dataset",            DATASET_NAME)
        mlflow.log_param("feature_set",        "morgan+maccs+rdkit_descriptors")
        mlflow.log_param("split_strategy",     "scaffold_grouped")
        mlflow.log_param("hp_search_method",   "optuna_tpe")
        mlflow.log_param("n_optuna_trials",    N_OPTUNA_TRIALS)
        mlflow.log_param("cv_folds",           N_CV_SPLITS)
        mlflow.log_param("cv_early_stopping",  True)
        mlflow.log_param("random_seed",        RANDOM_SEED)
        mlflow.log_param("ad_threshold",       AD_TANIMOTO_THRESHOLD)
        mlflow.log_param("quantile_models",    QUANTILE_MODELS_AVAILABLE)
        mlflow.log_param("target_unit",        TARGET_UNIT)

        for split_name, m in metrics_report.items():
            for metric_name, value in m.items():
                mlflow.log_metric(f"{split_name}_{metric_name}", value)

        for path in [
            model_path, fe_path, fps_path, metadata_path,
            metrics_path, trials_path, error_path, scaffold_error_path,
        ]:
            if path.exists():
                mlflow.log_artifact(str(path))

    logger.info("Logged to MLflow experiment: %s", MLFLOW_EXPERIMENT_NAME)
else:
    logger.warning("MLflow not available — skipping.")


## 18. Production inference contract

This is the exact function that maps to your FastAPI endpoint.

**Valid response:**
```json
{
  "status": "ok",
  "canonical_smiles": "CCO",
  "prediction": -0.42,
  "prediction_unit": "log(mol/L)",
  "prediction_interval_80": [-1.1, 0.2],
  "in_domain": true,
  "ad_score": 0.87,
  "model": "xgboost_aqsol_v3"
}
```

**Rejected response:**
```json
{
  "status": "rejected",
  "reason": "invalid_smiles",
  "original_smiles": "not_a_molecule"
}
```


In [ ]:
# Load from disk to simulate production cold-start
_model   = joblib.load(model_path)
_fe      = joblib.load(fe_path)
_fps     = joblib.load(fps_path)
_q10     = joblib.load(q10_path) if QUANTILE_MODELS_AVAILABLE and q10_path.exists() else None
_q90     = joblib.load(q90_path) if QUANTILE_MODELS_AVAILABLE and q90_path.exists() else None


def predict_solubility(smiles: Any) -> Dict[str, Any]:
    """
    Single-molecule inference with full production contract:
    - input validation
    - canonicalization
    - applicability domain check
    - point prediction
    - 80% prediction interval (if quantile models available)
    """
    validation = validate_smiles(smiles)

    if not validation.is_valid:
        logger.info("Rejected: %s | reason=%s", smiles, validation.rejection_reason)
        return {
            "status":          "rejected",
            "reason":          validation.rejection_reason,
            "original_smiles": str(smiles),
        }

    X = _fe.transform([validation.canonical_smiles])

    # Point estimate
    point = float(_model.predict(X)[0])

    # Prediction interval
    if _q10 is not None and _q90 is not None:
        lo = float(_q10.predict(X)[0])
        hi = float(_q90.predict(X)[0])
        interval = [round(lo, 4), round(hi, 4)]
    else:
        interval = None

    # Applicability domain
    ad_score = max_tanimoto_to_training(validation.canonical_smiles, _fps)
    in_domain = ad_score >= AD_TANIMOTO_THRESHOLD

    return {
        "status":                 "ok",
        "original_smiles":        validation.original_smiles,
        "canonical_smiles":       validation.canonical_smiles,
        "prediction":             round(point, 4),
        "prediction_unit":        TARGET_UNIT,
        "prediction_interval_80": interval,
        "in_domain":              in_domain,
        "ad_score":               round(ad_score, 4),
        "ad_threshold":           AD_TANIMOTO_THRESHOLD,
        "model":                  "xgboost_aqsol_v3",
    }


# Integration test
test_molecules = [
    "CCO",                           # ethanol — high solubility
    "CC(=O)Oc1ccccc1C(=O)O",        # aspirin
    "Cn1cnc2c1c(=O)n(C)c(=O)n2C",  # caffeine
    "not_a_smiles",                  # rejected
    "",                              # rejected
    None,                            # rejected
]

results_df = pd.DataFrame([predict_solubility(s) for s in test_molecules])
display(results_df)


## 19. FastAPI endpoint sketch

This cell is documentation only — do not run as a server inside the notebook.
Move to `src/api/main.py` and `src/models/predict.py`.


In [ ]:
FASTAPI_SKETCH = '''
# src/api/main.py
from typing import Optional
from pydantic import BaseModel
from fastapi import FastAPI
import joblib
from pathlib import Path

from src.models.predict import predict_solubility

app = FastAPI(
    title="CentaurDrug ADMET API",
    version="0.3.0",
)


class SolubilityRequest(BaseModel):
    smiles: str


class SolubilityResponse(BaseModel):
    status: str
    canonical_smiles: Optional[str] = None
    prediction: Optional[float] = None
    prediction_unit: Optional[str] = None
    prediction_interval_80: Optional[list] = None
    in_domain: Optional[bool] = None
    ad_score: Optional[float] = None
    reason: Optional[str] = None


@app.post("/admet/solubility", response_model=SolubilityResponse)
def predict_aqsol(request: SolubilityRequest) -> SolubilityResponse:
    result = predict_solubility(request.smiles)
    return SolubilityResponse(**result)


@app.get("/health")
def health():
    return {"status": "ok", "model": "xgboost_aqsol_v3"}
'''

print(FASTAPI_SKETCH)


## 20. What remains before true production

This notebook is a near-production prototype. What still needs to happen:

**Immediate (before first deployment):**
- Move all functions into `src/` modules
- Add `pytest` unit tests for `validate_smiles`, `predict_solubility`, feature extractor
- Wire FastAPI (`src/api/main.py`) and test with `uvicorn`
- Add Prometheus inference metrics (latency, rejection rate, out-of-domain rate)

**Short-term (before relying on predictions in decisions):**
- Add more ADMET endpoints (AMES, hERG, CYP450, BBB)
- Raise Optuna to 80–100 trials for final training run
- Consider conformal prediction (MAPIE) for statistically guaranteed intervals
- Add data version hash to metadata (DVC)

**Stretch:**
- Pretrained transformer baseline (ChemBERTa) for comparison on endpoints with >5k molecules
- Grafana dashboard for monitoring prediction drift
- Automated retraining pipeline (DVC + Airflow or Dagster)
